In [1]:
import glob

import xarray as xr
import numpy as np

import dask 

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.mpl
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

import xesmf as xe

In [2]:
from dask.distributed import Client

client = Client()
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.07/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45449 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/45449/status,
Dashboard: /proxy/45449/status,Workers: 7
Total threads: 7,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39447,Workers: 0
Dashboard: /proxy/45449/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35737,Total threads: 1
Dashboard: /proxy/46573/status,Memory: 4.57 GiB
Nanny: tcp://127.0.0.1:34765,


In [3]:
def detrend_1d(y, deg=1):
    x = np.arange(y.size)  
    z = np.polyfit(x, y, deg)  
    trend = np.polyval(z, x) 
    return y - trend + np.mean(y)

def detrend_monthly(da, deg=1):
    return da.groupby('time.month').apply(
        lambda x: xr.apply_ufunc(
            detrend_1d, x,
            input_core_dims=[['time']],
            output_core_dims=[['time']],
            vectorize=True,
            dask='parallelized',
            output_dtypes=[float],
            kwargs={'deg': deg}
        )
    )
    
def get_seasonal_means(da, seasons):
    da_season_mean = {}
    for season in seasons:
        if season == 'DJF':
            da_season_tmp = da.where(da['time.season'] == 'DJF')
            da_season = da_season_tmp.rolling(min_periods=3, center=True, time=3).mean()
            da_season_mean[season] = da_season.groupby('time.year').mean(dim='time')
        else:
            da_season = da.where(da['time.season'] == season, drop=True)
            da_season_mean[season] = da_season.groupby('time.year').mean(dim='time')
    return da_season_mean

def convert_units(da):
    # Convert Kelvin to Celsius
    if da.attrs.get('units') == 'K':
        da = da - 273.15
        da.attrs['units'] = '°C' 
    
    # Convert Pascal to hectoPascal
    elif da.attrs.get('units') == 'Pa':
        da = da / 100.
        da.attrs['units'] = 'hPa' 

    # Convert kg m-2 s-1 to mm/day
    elif da.attrs.get('units') == 'kg m-2 s-1':
        da = da*86400.
        da.attrs['units'] = 'mm/day' 
    return da

<span style="color: blue; font-size: 18px; font-weight: bold;">1) Read in variables for the last 200 years of each simulation</span>

In [4]:
import warnings
warnings.filterwarnings('ignore')

exp_ts = convert_units(xr.open_dataset("/scratch/p66/fc6164/MOPPeR/ts_Amon_ACCESS-ESM1-5_piControl_r1i1p1f1_gn_159201-209112.nc", decode_times=True).sel(time=slice('1800','2000')).ts)
exp_pr = convert_units(xr.open_dataset("/scratch/p66/fc6164/MOPPeR/pr_Amon_ACCESS-ESM1-5_piControl_r1i1p1f1_gn_159201-209112.nc", decode_times=True).sel(time=slice('1800','2000')).pr)

exp_ts = exp_ts.transpose('time', 'lat', 'lon')
exp_pr = exp_pr.transpose('time', 'lat', 'lon')

In [5]:
esm15_dir = "/scratch/eg3/zg0866/"
esm15_ts = convert_units(xr.open_dataset(esm15_dir+"ts_Amon_ACCESS-ESM1-5_piControl_r1i1p1f1_gn_010101-110012.nc").ts.isel(time=slice(-2400, None))).chunk({'time': -1}).transpose('lat', 'lon', 'time') 
esm15_pr = convert_units(xr.open_mfdataset(esm15_dir+"pr_Amon_ACCESS-ESM1-5_piControl_r1i1p1f1_gn_010101-110012.nc").pr.isel(time=slice(-2400, None))).chunk({'time': -1}).transpose('lat', 'lon', 'time')

<span style="color: blue; font-size: 18px; font-weight: bold;">2) Detrend</span>

In [6]:
exp_ts_dt = detrend_monthly(exp_ts, deg=1).compute()
exp_pr_dt = detrend_monthly(exp_pr, deg=1).compute()

In [7]:
esm15_ts_dt = detrend_monthly(esm15_ts, deg=1).compute()
esm15_pr_dt = detrend_monthly(esm15_pr, deg=1).compute()

<span style="color: blue; font-size: 18px; font-weight: bold;">3) Calculate seasonal means</span>

In [ ]:
exp_ts_ann = exp_ts_dt.groupby('time.year').mean(dim='time')
exp_ts_seas = get_seasonal_means(exp_ts_dt,['DJF','JJA'])

exp_pr_ann = exp_pr_dt.groupby('time.year').mean(dim='time')
exp_pr_seas = get_seasonal_means(exp_pr_dt,['DJF','JJA'])

esm15_ts_ann = esm15_ts_dt.groupby('time.year').mean(dim='time')
esm15_ts_seas = get_seasonal_means(esm15_ts_dt,['DJF','JJA'])

esm15_pr_ann = esm15_pr_dt.groupby('time.year').mean(dim='time')
esm15_pr_seas = get_seasonal_means(esm15_pr_dt,['DJF','JJA'])

<span style="color: blue; font-size: 18px; font-weight: bold;">4) Observations comparison</span>

In [ ]:
gpcp_dir = "/g/data/eg3/zg0866/plots/ACCESS_eval/red-run/"
gpcp_pr = xr.open_dataset(gpcp_dir+"precip.mon.mean.nc").precip.sel(time=slice('1979','2022'))

pr_obs_regridder = xe.Regridder(gpcp_pr, exp_pr, method='bilinear')
pr_obs_regridded = pr_obs_regridder(gpcp_pr)

pr_obs_dt = detrend_monthly(pr_obs_regridded, deg=1).compute()

pr_obs_ann = pr_obs_dt.groupby('time.year').mean(dim='time')
pr_obs_seas = get_seasonal_means(pr_obs_dt,['DJF','JJA'])

In [ ]:
#dir = "/g/data/rt52/era5/single-levels/monthly-averaged/skt/*/skt_era5_moda_sfc_*.nc"
#files = glob.glob(dir)

# Filter for years 1979–2022
#filtered_files = [
#    f for f in files
#    if 1979 <= int(re.search(r'sfc_(\d{4})', f).group(1)) <= 2022
#]

#era5_ts = convert_units(xr.open_mfdataset(
#    filtered_files,
#    combine='by_coords',
#    parallel=True,
#    chunks={'time': 12}
#).skt)

# Trigger actual data load
#era5_ts.load()

era5_ts = xr.open_dataset("/scratch/eg3/zg0866/era5_skt_1979_2022.nc").skt

# Regrid and compute immediately
ts_obs_regridder = xe.Regridder(era5_ts, exp_ts, method='bilinear')
ts_obs_regridded = ts_obs_regridder(era5_ts).compute()

ts_obs_dt = detrend_monthly(ts_obs_regridded, deg=1)

ts_obs_ann = ts_obs_dt.groupby('time.year').mean(dim='time')
ts_obs_seas = get_seasonal_means(ts_obs_dt,['DJF','JJA'])

<span style="color: blue; font-size: 18px; font-weight: bold;">5) Plot</span>

In [ ]:
observations = xr.concat([pr_obs_seas['DJF'],
                         pr_obs_seas['JJA']], dim='seas')
observations['seas'] = ['DJF','JJA']

esm15 = xr.concat([esm15_pr_seas['DJF'], 
                         esm15_pr_seas['JJA']], dim='seas')
esm15['seas'] = ['DJF','JJA']

test_run = xr.concat([exp_pr_seas['DJF'], 
                         exp_pr_seas['JJA']], dim='seas')
test_run['seas'] = ['DJF','JJA']

cmaps = [plt.get_cmap('YlGnBu'),  # Observations
         plt.get_cmap('YlGnBu'),   # ESM1.5
         plt.get_cmap('YlGnBu'),  # Test Run
         plt.get_cmap('RdBu')]   # Test - ESM1.5
extend_opts = ['max', 'max', 'max', 'both']  

vmin_main, vmax_main = 0, 16
vmin_diff, vmax_diff = -3, 3
levels_main = np.linspace(vmin_main, vmax_main, 21)
levels_diff = np.linspace(vmin_diff, vmax_diff, 21)
season_labels = ['DJF', 'JJA']
column_titles = ['Observations (GPCC)', 'ESM1.5', 'Test Run', 'Test Run - ESM1.5']
n_seasons = observations.shape[0]  

fig, ax = plt.subplots(n_seasons, 4, figsize=(22, 8), sharex=True, sharey=True,
    subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

ax = ax.reshape(n_seasons, 4)
obs_fields   = [observations[i].mean(dim='year').compute() for i in range(n_seasons)]
esm_fields   = [esm15[i].mean(dim='year').compute() for i in range(n_seasons)]
run_fields   = [test_run[i].mean(dim='year').compute() for i in range(n_seasons)]
diff_fields  = [run_fields[i] - esm_fields[i] for i in range(n_seasons)]
diff_fields  = [field.compute() for field in diff_fields]

fields_per_col = [obs_fields, esm_fields, run_fields, diff_fields]
vmins = [vmin_main, vmin_main, vmin_main, vmin_diff]
vmaxs = [vmax_main, vmax_main, vmax_main, vmax_diff]
levels = [levels_main, levels_main, levels_main, levels_diff]

mappables = [[], [], [], []]

for i in range(n_seasons):
    for j in range(4):
        mappable = fields_per_col[j][i].plot.contourf(
            ax=ax[i, j],
            cmap=cmaps[j],
            vmin=vmins[j],
            vmax=vmaxs[j],
            levels=levels[j],
            add_colorbar=False,
            extend=extend_opts[j],
            transform=ccrs.PlateCarree()
        )

        mappables[j].append(mappable)

        if i == 0:
            ax[i, j].set_title(column_titles[j], fontsize=16, fontweight='bold')
        else:
            ax[i, j].set_title('')
        if j == 0:
            ax[i, j].set_ylabel(season_labels[i], fontsize=16, fontweight='bold')
        else:
            ax[i, j].set_ylabel('')
for row in ax:
    for axi in row:
        axi.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True, degree_symbol=''))
        axi.yaxis.set_major_formatter(LatitudeFormatter(degree_symbol=''))
        axi.coastlines('110m')
        axi.set_xticks(np.arange(-180, 181, 60))
        axi.set_yticks(np.arange(-90, 91, 45))
        axi.set_xlabel('')
        axi.tick_params(axis='both', labelsize=12) 

plt.subplots_adjust(wspace=0.2, hspace=0.16, bottom=0.13, top=0.94)

for j in range(4):
    cbar = fig.colorbar(
        mappables[j][0],
        ax=ax[:, j],
        orientation='horizontal',
        fraction=0.06,
        pad=0.1
    )
    cbar.set_label(
        column_titles[j] + (" (mm/day)"),
        fontsize=14
    )
    cbar.ax.tick_params(labelsize=12) 
    
#plt.savefig("/g/data/eg3/zg0866/plots/ACCESS_eval/pr_obs_esm15_testrun_djf_jja.png", bbox_inches="tight", dpi=750)

In [ ]:
observations = xr.concat([ts_obs_seas['DJF'],
                         ts_obs_seas['JJA']], dim='seas')
observations['seas'] = ['DJF','JJA']

esm15 = xr.concat([esm15_ts_seas['DJF'], 
                         esm15_ts_seas['JJA']], dim='seas')
esm15['seas'] = ['DJF','JJA']

test_run = xr.concat([exp_ts_seas['DJF'], 
                         exp_ts_seas['JJA']], dim='seas')
test_run['seas'] = ['DJF','JJA']

cmaps = [plt.get_cmap('coolwarm'),  # Observations
         plt.get_cmap('coolwarm'),   # ESM1.5
         plt.get_cmap('coolwarm'),  # Test Run
         plt.get_cmap('RdBu_r')]   # Test - ESM1.5
extend_opts = ['both', 'both', 'both', 'both']  

vmin_main, vmax_main = -32, 32
vmin_diff, vmax_diff = -5, 5
levels_main = np.linspace(vmin_main, vmax_main, 21)
levels_diff = np.linspace(vmin_diff, vmax_diff, 21)
season_labels = ['DJF', 'JJA']
column_titles = ['Observations (ERA5)', 'ESM1.5', 'Test Run', 'Test Run - ESM1.5']
n_seasons = observations.shape[0]  

fig, ax = plt.subplots(n_seasons, 4, figsize=(22, 8), sharex=True, sharey=True,
    subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

ax = ax.reshape(n_seasons, 4)
obs_fields   = [observations[i].mean(dim='year').compute() for i in range(n_seasons)]
esm_fields   = [esm15[i].mean(dim='year').compute() for i in range(n_seasons)]
run_fields   = [test_run[i].mean(dim='year').compute() for i in range(n_seasons)]
diff_fields  = [run_fields[i] - esm_fields[i] for i in range(n_seasons)]
diff_fields  = [field.compute() for field in diff_fields]

fields_per_col = [obs_fields, esm_fields, run_fields, diff_fields]
vmins = [vmin_main, vmin_main, vmin_main, vmin_diff]
vmaxs = [vmax_main, vmax_main, vmax_main, vmax_diff]
levels = [levels_main, levels_main, levels_main, levels_diff]

mappables = [[], [], [], []]

for i in range(n_seasons):
    for j in range(4):
        mappable = fields_per_col[j][i].plot.contourf(
            ax=ax[i, j],
            cmap=cmaps[j],
            vmin=vmins[j],
            vmax=vmaxs[j],
            levels=levels[j],
            add_colorbar=False,
            extend=extend_opts[j],
            transform=ccrs.PlateCarree()
        )

        mappables[j].append(mappable)

        if i == 0:
            ax[i, j].set_title(column_titles[j], fontsize=16, fontweight='bold')
        else:
            ax[i, j].set_title('')
        if j == 0:
            ax[i, j].set_ylabel(season_labels[i], fontsize=16, fontweight='bold')
        else:
            ax[i, j].set_ylabel('')
for row in ax:
    for axi in row:
        axi.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True, degree_symbol=''))
        axi.yaxis.set_major_formatter(LatitudeFormatter(degree_symbol=''))
        axi.coastlines('110m')
        axi.set_xticks(np.arange(-180, 181, 60))
        axi.set_yticks(np.arange(-90, 91, 45))
        axi.set_xlabel('')
        axi.tick_params(axis='both', labelsize=12) 

plt.subplots_adjust(wspace=0.2, hspace=0.16, bottom=0.13, top=0.94)

for j in range(4):
    cbar = fig.colorbar(
        mappables[j][0],
        ax=ax[:, j],
        orientation='horizontal',
        fraction=0.06,
        pad=0.1
    )
    cbar.set_label(
        column_titles[j] + (" (°C)"),
        fontsize=14
    )
    cbar.ax.tick_params(labelsize=12) 
    
#plt.savefig("/g/data/eg3/zg0866/plots/ACCESS_eval/ts_obs_esm15_testrun_djf_jja.png", bbox_inches="tight", dpi=750)